In [2]:
"""
Multi-Seed Aggregation (Phases 6-12) -- COLAB VERSION.

Identical logic to Q1_Revision/scripts/multiseed_aggregate.py, only the
BASE/OUT paths differ (Google Drive instead of local D:). Paste into a new
Colab cell and run standalone -- it does NOT need Notebook 05/07 objects in
scope, only Drive mounted.

Expects (per MULTISEED_EXPERIMENT_PLAN.md's two-line diff, already applied
when you trained seed123/seed2024):
  /content/drive/MyDrive/ROCOv2/Prepared/Baselines/DenseNet121_Unimodal/                 <- seed 42 (reference)
  /content/drive/MyDrive/ROCOv2/Prepared/Baselines/DenseNet121_Unimodal_seed123/
  /content/drive/MyDrive/ROCOv2/Prepared/Baselines/DenseNet121_Unimodal_seed2024/
  /content/drive/MyDrive/ROCOv2/Prepared/Baselines/BiomedCLIP_VLM_Trained/                <- seed 42 (reference)
  /content/drive/MyDrive/ROCOv2/Prepared/Baselines/BiomedCLIP_VLM_Trained_seed123/
  /content/drive/MyDrive/ROCOv2/Prepared/Baselines/BiomedCLIP_VLM_Trained_seed2024/

Outputs -> /content/drive/MyDrive/ROCOv2/Prepared/multiseed_validation/ (on Drive,
so it persists after the runtime disconnects):
  multiseed_results_all.csv, multiseed_summary_statistics.csv,
  multiseed_per_label_results.csv, multiseed_model_comparison.csv,
  multiseed_reproducibility_check.csv, multiseed_training_metadata.csv,
  multiseed_validation_report.md
"""
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (precision_score, recall_score, f1_score, hamming_loss,
                              roc_auc_score, average_precision_score)

BASE = Path("/content/drive/MyDrive/ROCOv2/Prepared")
BASELINES_DIR = BASE / "Baselines"
OUT = BASE / "multiseed_validation"
OUT.mkdir(parents=True, exist_ok=True)

with open(BASE / "dataset_config.json") as f:
    LABEL_COLUMNS = json.load(f)["label_columns"]
with open(BASE / "reproducibility.json") as f:
    TAXO = json.load(f)["label_taxonomy"]
LABEL_NAMES = [TAXO[c.replace("label_", "")] for c in LABEL_COLUMNS]

test_final_ids = set(pd.read_csv(BASE / "test_final_prepared.csv", usecols=["ID"])["ID"])

# (reference_dir_name, seed_prefix_pattern, predictions_filename, config_filename, display_name)
MODEL_SPECS = [
    ("DenseNet121_Unimodal", r"^DenseNet121_Unimodal_seed(\d+)$",
     "densenet121_predictions.csv", "densenet121_config.json", "DenseNet121 (image-only baseline)"),
    ("BiomedCLIP_VLM_Trained", r"^BiomedCLIP_VLM_Trained_seed(\d+)$",
     "vlm_predictions.csv", "vlm_config.json", "Proposed VLM (fine-tuned BiomedCLIP)"),
]


def discover_seed_dirs(reference_name, pattern):
    """Returns {seed: dir_path}, including the reference dir under seed 42."""
    found = {42: BASELINES_DIR / reference_name}
    pat = re.compile(pattern)
    if BASELINES_DIR.exists():
        for d in BASELINES_DIR.iterdir():
            m = pat.match(d.name)
            if m:
                found[int(m.group(1))] = d
    return {s: d for s, d in found.items() if d.exists()}


def compute_metrics(y_true, y_score, y_pred):
    macro_p = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_r = recall_score(y_true, y_pred, average="macro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    micro_p = precision_score(y_true, y_pred, average="micro", zero_division=0)
    micro_r = recall_score(y_true, y_pred, average="micro", zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    hl = hamming_loss(y_true, y_pred)
    subset_acc = float((y_true == y_pred).all(axis=1).mean())
    per_label_roc, per_label_pr = [], []
    for j in range(y_true.shape[1]):
        t, s = y_true[:, j], y_score[:, j]
        if 0 < t.sum() < len(t):
            per_label_roc.append(roc_auc_score(t, s))
            per_label_pr.append(average_precision_score(t, s))
    macro_roc = float(np.nanmean(per_label_roc)) if per_label_roc else np.nan
    macro_pr = float(np.nanmean(per_label_pr)) if per_label_pr else np.nan
    micro_roc = roc_auc_score(y_true.ravel(), y_score.ravel())
    micro_pr = average_precision_score(y_true.ravel(), y_score.ravel())
    return {
        "Micro_Precision": micro_p, "Micro_Recall": micro_r, "Micro_F1": micro_f1,
        "Macro_Precision": macro_p, "Macro_Recall": macro_r, "Macro_F1": macro_f1,
        "Hamming_Loss": hl, "Subset_Accuracy": subset_acc,
        "ROC_AUC_Macro": macro_roc, "ROC_AUC_Micro": micro_roc,
        "PR_AUC_Macro": macro_pr, "PR_AUC_Micro": micro_pr,
    }


def load_predictions(path):
    df = pd.read_csv(path)
    df = df[df["ID"].isin(test_final_ids)].sort_values("ID").reset_index(drop=True)
    if len(df) != 4935:
        raise ValueError(f"{path}: expected 4935 Test-final rows, got {len(df)} -- "
                          "do not silently proceed with a mismatched split.")
    y_true = np.stack([df[f"{c}_true"].values for c in LABEL_COLUMNS], axis=1)
    y_score = np.stack([df[f"{c}_prob"].values for c in LABEL_COLUMNS], axis=1)
    y_pred = np.stack([df[f"{c}_pred"].values for c in LABEL_COLUMNS], axis=1)
    return df, y_true, y_score, y_pred


all_rows = []
per_label_rows = []
metadata_rows = []
per_model_arrays = {}  # (display_name, seed) -> (y_true, y_score, y_pred)

for reference_name, pattern, pred_file, config_file, display_name in MODEL_SPECS:
    seed_dirs = discover_seed_dirs(reference_name, pattern)
    print(f"{display_name}: found {len(seed_dirs)} seed(s): {sorted(seed_dirs.keys())}")

    for seed, d in sorted(seed_dirs.items()):
        pred_path = d / pred_file
        if not pred_path.exists():
            print(f"  WARNING: {pred_path} missing -- skipping seed {seed} for {display_name}.")
            continue
        df, y_true, y_score, y_pred = load_predictions(pred_path)
        metrics = compute_metrics(y_true, y_score, y_pred)
        row = {"Model": display_name, "Seed": seed, "N_Test_Final": len(df), **metrics}
        all_rows.append(row)
        per_model_arrays[(display_name, seed)] = (y_true, y_score, y_pred)

        for j, col in enumerate(LABEL_COLUMNS):
            f1 = f1_score(y_true[:, j], y_pred[:, j], zero_division=0)
            per_label_rows.append({"Model": display_name, "Seed": seed,
                                    "Label_Name": LABEL_NAMES[j], "F1": f1})

        config_path = d / config_file
        repro_path = d / "reproducibility.json"
        meta = {"Model": display_name, "Seed": seed, "Directory": str(d)}
        if config_path.exists():
            with open(config_path) as f:
                cfg = json.load(f)
            tc = cfg.get("training_config", {})
            meta.update({
                "Batch_Size": tc.get("batch_size"), "Stage1_Max_Epochs": tc.get("stage1_max_epochs"),
                "Stage2_Max_Epochs": tc.get("stage2_max_epochs"), "Early_Stopping_Patience": tc.get("early_stopping_patience"),
                "Decision_Threshold": tc.get("decision_threshold"),
            })
        if repro_path.exists():
            with open(repro_path) as f:
                repro = json.load(f)
            meta.update({
                "Python_Version": repro.get("python_version"), "PyTorch_Version": repro.get("pytorch_version"),
                "CUDA_Version": repro.get("cuda_version"), "GPU": repro.get("gpu_name"),
                "Best_Epoch": (repro.get("best_epoch") or {}).get("epoch"),
                "Best_Val_Macro_F1": (repro.get("best_epoch") or {}).get("val_macro_f1"),
            })
        metadata_rows.append(meta)

results_df = pd.DataFrame(all_rows)
results_df.to_csv(OUT / "multiseed_results_all.csv", index=False)

per_label_df = pd.DataFrame(per_label_rows)
per_label_df.to_csv(OUT / "multiseed_per_label_results.csv", index=False)

metadata_df = pd.DataFrame(metadata_rows)
metadata_df.to_csv(OUT / "multiseed_training_metadata.csv", index=False)

# --- Phase 7: summary statistics across seeds ---
summary_rows = []
metric_cols = ["Micro_Precision", "Micro_Recall", "Micro_F1", "Macro_Precision", "Macro_Recall",
               "Macro_F1", "Hamming_Loss", "Subset_Accuracy", "ROC_AUC_Macro", "ROC_AUC_Micro",
               "PR_AUC_Macro", "PR_AUC_Micro"]
for display_name in results_df["Model"].unique():
    sub = results_df[results_df["Model"] == display_name]
    n_seeds = len(sub)
    for metric in metric_cols:
        vals = sub[metric].values
        mean, std = float(np.mean(vals)), float(np.std(vals, ddof=1)) if n_seeds > 1 else np.nan
        if n_seeds > 1:
            sem = std / np.sqrt(n_seeds)
            ci_lo, ci_hi = mean - 1.96 * sem, mean + 1.96 * sem
        else:
            ci_lo, ci_hi = np.nan, np.nan
        summary_rows.append({
            "Model": display_name, "Metric": metric, "N_Seeds": n_seeds,
            "Mean": mean, "SD": std, "Min": float(np.min(vals)), "Max": float(np.max(vals)),
            "CI95_Lower_normal_approx": ci_lo, "CI95_Upper_normal_approx": ci_hi,
            "Note": "SD/CI undefined with a single seed -- do not report as if characterizing variability" if n_seeds < 2 else "",
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT / "multiseed_summary_statistics.csv", index=False)

# --- Phase 8: Proposed VLM - DenseNet121 per-seed comparison ---
comparison_rows = []
vlm_name = "Proposed VLM (fine-tuned BiomedCLIP)"
dn_name = "DenseNet121 (image-only baseline)"
vlm_seeds = set(results_df[results_df["Model"] == vlm_name]["Seed"])
dn_seeds = set(results_df[results_df["Model"] == dn_name]["Seed"])
common_seeds = sorted(vlm_seeds & dn_seeds)
for metric in ["Macro_F1", "Micro_F1", "ROC_AUC_Macro", "PR_AUC_Macro"]:
    diffs = []
    for seed in common_seeds:
        v = results_df[(results_df["Model"] == vlm_name) & (results_df["Seed"] == seed)][metric].iloc[0]
        d = results_df[(results_df["Model"] == dn_name) & (results_df["Seed"] == seed)][metric].iloc[0]
        diffs.append({"Seed": seed, "Metric": metric, "VLM": v, "DenseNet121": d, "Difference": v - d})
    comparison_rows.extend(diffs)
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(OUT / "multiseed_model_comparison.csv", index=False)

comparison_summary = (
    comparison_df.groupby("Metric")["Difference"]
    .agg(Mean_Improvement="mean", SD="std", Min_Improvement="min", Max_Improvement="max", N_Seeds="count")
    .reset_index()
) if len(comparison_df) else pd.DataFrame()

# --- Phase 9: seed-42 reproducibility check (only meaningful if seed 42 was actually retrained
#     into a *_seed42 directory -- the plain reference directories above are NOT a retraining) ---
repro_check_rows = []
for reference_name, pattern, pred_file, config_file, display_name in MODEL_SPECS:
    retrained_42_dir = BASELINES_DIR / f"{reference_name}_seed42"
    reference_dir = BASELINES_DIR / reference_name
    if retrained_42_dir.exists() and (retrained_42_dir / pred_file).exists():
        _, yt_new, ys_new, yp_new = load_predictions(retrained_42_dir / pred_file)
        _, yt_ref, ys_ref, yp_ref = load_predictions(reference_dir / pred_file)
        m_new = compute_metrics(yt_new, ys_new, yp_new)
        m_ref = compute_metrics(yt_ref, ys_ref, yp_ref)
        for metric in metric_cols:
            diff = abs(m_new[metric] - m_ref[metric])
            repro_check_rows.append({
                "Model": display_name, "Metric": metric,
                "Original_Seed42_Value": m_ref[metric], "Retrained_Seed42_Value": m_new[metric],
                "Absolute_Difference": diff,
                "Status": "PASS" if diff < 1e-4 else "DIFFERS -- investigate (nondeterminism on GPU is possible even with seeding)",
            })
    else:
        repro_check_rows.append({
            "Model": display_name, "Metric": "(all)", "Original_Seed42_Value": "N/A",
            "Retrained_Seed42_Value": "N/A", "Absolute_Difference": "N/A",
            "Status": "SEED 42 NOT RETRAINED -- original result retained as reference, per plan. "
                      "This is not a failure; it is the default per the experiment plan unless you "
                      "explicitly chose to also retrain seed 42.",
        })
repro_check_df = pd.DataFrame(repro_check_rows)
repro_check_df.to_csv(OUT / "multiseed_reproducibility_check.csv", index=False)

# --- Phase 11: report ---
report = ["# Multi-Seed Validation Report\n\n"]
for display_name in results_df["Model"].unique() if len(results_df) else []:
    n = len(results_df[results_df["Model"] == display_name])
    report.append(f"- **{display_name}**: {n} seed(s) with usable Test-final predictions found.\n")

if len(results_df) == 0:
    report.append("\n**No seed-suffixed model directories were found yet.** This report reflects an "
                   "empty aggregation run -- re-run this script after training completes for at least "
                   "one additional seed. No results are fabricated in their absence.\n")
else:
    report.append("\n## Phase 11 questions\n\n")
    for display_name in results_df["Model"].unique():
        sub = results_df[results_df["Model"] == display_name]
        n = len(sub)
        if n < 2:
            report.append(f"### {display_name}\nOnly {n} seed available -- stability across seeds "
                           "cannot be assessed yet. Train at least one more seed before drawing any "
                           "conclusion about robustness for this model.\n\n")
            continue
        macro_f1_vals = sub["Macro_F1"].values
        cv = np.std(macro_f1_vals, ddof=1) / np.mean(macro_f1_vals)
        report.append(f"### {display_name} ({n} seeds)\n")
        report.append(f"- Macro-F1 across seeds: mean={np.mean(macro_f1_vals):.4f}, "
                       f"SD={np.std(macro_f1_vals, ddof=1):.4f}, "
                       f"range=[{np.min(macro_f1_vals):.4f}, {np.max(macro_f1_vals):.4f}], "
                       f"coefficient of variation={cv:.4f}\n")
        if 42 in sub["Seed"].values:
            seed42_val = sub[sub["Seed"] == 42]["Macro_F1"].iloc[0]
            within_range = np.min(macro_f1_vals) <= seed42_val <= np.max(macro_f1_vals)
            report.append(f"- Original seed-42 Macro-F1 ({seed42_val:.4f}) falls "
                           f"{'WITHIN' if within_range else 'OUTSIDE'} the observed multi-seed range.\n")
        report.append("\n")

    if len(comparison_summary):
        report.append("## Proposed VLM vs. DenseNet121 (Phase 8)\n\n")
        report.append(comparison_summary.to_string(index=False))
        report.append("\n\n")
        all_positive = (comparison_df.groupby("Metric")["Difference"].min() > 0).all()
        report.append(f"Proposed VLM improvement over DenseNet121 is "
                       f"{'consistently positive across every available seed and metric' if all_positive else 'NOT consistently positive across all seeds/metrics -- see per-seed table'}.\n")
        if len(common_seeds) < 3:
            report.append(f"\nCaution: only {len(common_seeds)} seed(s) have both models available -- "
                           "do not overclaim statistical significance from this few points.\n")

with open(OUT / "multiseed_validation_report.md", "w") as f:
    f.writelines(str(x) for x in report)

print("".join(str(x) for x in report))
print(f"\nOutputs written to {OUT}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DenseNet121 (image-only baseline): found 3 seed(s): [42, 123, 2024]
Proposed VLM (fine-tuned BiomedCLIP): found 3 seed(s): [42, 123, 2024]
# Multi-Seed Validation Report

- **DenseNet121 (image-only baseline)**: 3 seed(s) with usable Test-final predictions found.
- **Proposed VLM (fine-tuned BiomedCLIP)**: 3 seed(s) with usable Test-final predictions found.

## Phase 11 questions

### DenseNet121 (image-only baseline) (3 seeds)
- Macro-F1 across seeds: mean=0.7212, SD=0.0047, range=[0.7170, 0.7263], coefficient of variation=0.0066
- Original seed-42 Macro-F1 (0.7170) falls WITHIN the observed multi-seed range.

### Proposed VLM (fine-tuned BiomedCLIP) (3 seeds)
- Macro-F1 across seeds: mean=0.8370, SD=0.0052, range=[0.8311, 0.8407], coefficient of variation=0.0062
- Original seed-42 Macro-F1 (0.8311) falls WITHIN the observed multi-seed range.

## Proposed VL

In [3]:
"""
Prints the full contents of every CSV multiseed_aggregate_COLAB.py already wrote,
so the results can be pasted into chat for the verification report. Run this
AFTER multiseed_aggregate_COLAB.py, in the same or a new Colab cell -- it only
reads back files from Drive, no recomputation.
"""
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

OUT = Path("/content/drive/MyDrive/ROCOv2/Prepared/multiseed_validation")

for fname in [
    "multiseed_results_all.csv",
    "multiseed_summary_statistics.csv",
    "multiseed_per_label_results.csv",
    "multiseed_model_comparison.csv",
    "multiseed_reproducibility_check.csv",
    "multiseed_training_metadata.csv",
]:
    print(f"\n{'=' * 100}\n{fname}\n{'=' * 100}")
    df = pd.read_csv(OUT / fname)
    print(df.to_string(index=False))



multiseed_results_all.csv
                               Model  Seed  N_Test_Final  Micro_Precision  Micro_Recall  Micro_F1  Macro_Precision  Macro_Recall  Macro_F1  Hamming_Loss  Subset_Accuracy  ROC_AUC_Macro  ROC_AUC_Micro  PR_AUC_Macro  PR_AUC_Micro
   DenseNet121 (image-only baseline)    42          4935           0.8216        0.8973    0.8578           0.6879        0.7685    0.7170        0.0260           0.6646         0.9697         0.9888        0.7788        0.9358
   DenseNet121 (image-only baseline)   123          4935           0.8495        0.8867    0.8677           0.7076        0.7566    0.7263        0.0237           0.6993         0.9665         0.9871        0.7877        0.9346
   DenseNet121 (image-only baseline)  2024          4935           0.8229        0.8958    0.8578           0.6922        0.7668    0.7203        0.0260           0.6679         0.9690         0.9882        0.7808        0.9342
Proposed VLM (fine-tuned BiomedCLIP)    42          4935     

In [4]:
"""
Per-label ROC-AUC / PR-AUC follow-up -- COLAB VERSION.

Fills the gap flagged in the verification report: multiseed_aggregate_COLAB.py
only computed per-label F1. This script recomputes per-label F1, ROC-AUC, and
PR-AUC together, using the identical seed-discovery/loading logic (same
Test-final filter, same 4935-row hard check), and writes one combined CSV.

No retraining, no changes to existing outputs -- purely an additional read
pass over the same prediction files multiseed_aggregate_COLAB.py already used.

Run in Colab (Drive mounted, same as before).

Output -> /content/drive/MyDrive/ROCOv2/Prepared/multiseed_validation/
  multiseed_per_label_results_full.csv  (Model, Seed, Label_Name, F1, ROC_AUC, PR_AUC)
"""
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

BASE = Path("/content/drive/MyDrive/ROCOv2/Prepared")
BASELINES_DIR = BASE / "Baselines"
OUT = BASE / "multiseed_validation"
OUT.mkdir(parents=True, exist_ok=True)

with open(BASE / "dataset_config.json") as f:
    LABEL_COLUMNS = json.load(f)["label_columns"]
with open(BASE / "reproducibility.json") as f:
    TAXO = json.load(f)["label_taxonomy"]
LABEL_NAMES = [TAXO[c.replace("label_", "")] for c in LABEL_COLUMNS]

test_final_ids = set(pd.read_csv(BASE / "test_final_prepared.csv", usecols=["ID"])["ID"])

MODEL_SPECS = [
    ("DenseNet121_Unimodal", r"^DenseNet121_Unimodal_seed(\d+)$",
     "densenet121_predictions.csv", "DenseNet121 (image-only baseline)"),
    ("BiomedCLIP_VLM_Trained", r"^BiomedCLIP_VLM_Trained_seed(\d+)$",
     "vlm_predictions.csv", "Proposed VLM (fine-tuned BiomedCLIP)"),
]


def discover_seed_dirs(reference_name, pattern):
    found = {42: BASELINES_DIR / reference_name}
    pat = re.compile(pattern)
    if BASELINES_DIR.exists():
        for d in BASELINES_DIR.iterdir():
            m = pat.match(d.name)
            if m:
                found[int(m.group(1))] = d
    return {s: d for s, d in found.items() if d.exists()}


def load_predictions(path):
    df = pd.read_csv(path)
    df = df[df["ID"].isin(test_final_ids)].sort_values("ID").reset_index(drop=True)
    if len(df) != 4935:
        raise ValueError(f"{path}: expected 4935 Test-final rows, got {len(df)}")
    y_true = np.stack([df[f"{c}_true"].values for c in LABEL_COLUMNS], axis=1)
    y_score = np.stack([df[f"{c}_prob"].values for c in LABEL_COLUMNS], axis=1)
    y_pred = np.stack([df[f"{c}_pred"].values for c in LABEL_COLUMNS], axis=1)
    return y_true, y_score, y_pred


rows = []
for reference_name, pattern, pred_file, display_name in MODEL_SPECS:
    seed_dirs = discover_seed_dirs(reference_name, pattern)
    print(f"{display_name}: found {len(seed_dirs)} seed(s): {sorted(seed_dirs.keys())}")
    for seed, d in sorted(seed_dirs.items()):
        pred_path = d / pred_file
        if not pred_path.exists():
            print(f"  WARNING: {pred_path} missing -- skipping seed {seed}.")
            continue
        y_true, y_score, y_pred = load_predictions(pred_path)
        for j, label_name in enumerate(LABEL_NAMES):
            t, s, p = y_true[:, j], y_score[:, j], y_pred[:, j]
            f1 = f1_score(t, p, zero_division=0)
            if 0 < t.sum() < len(t):
                roc = roc_auc_score(t, s)
                pr = average_precision_score(t, s)
            else:
                roc, pr = np.nan, np.nan
            rows.append({"Model": display_name, "Seed": seed, "Label_Name": label_name,
                         "F1": f1, "ROC_AUC": roc, "PR_AUC": pr,
                         "N_Positive": int(t.sum())})

result_df = pd.DataFrame(rows)
result_df.to_csv(OUT / "multiseed_per_label_results_full.csv", index=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print("\n" + result_df.to_string(index=False))
print(f"\nSaved {len(result_df)} rows to {OUT / 'multiseed_per_label_results_full.csv'}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DenseNet121 (image-only baseline): found 3 seed(s): [42, 123, 2024]
Proposed VLM (fine-tuned BiomedCLIP): found 3 seed(s): [42, 123, 2024]

                               Model  Seed                   Label_Name     F1  ROC_AUC  PR_AUC  N_Positive
   DenseNet121 (image-only baseline)    42                      Abdomen 0.4196   0.8823  0.4887         276
   DenseNet121 (image-only baseline)    42                    Angiogram 0.7537   0.9936  0.8187         210
   DenseNet121 (image-only baseline)    42                       Breast 0.5185   0.9271  0.4989          18
   DenseNet121 (image-only baseline)    42              Lower Extremity 0.8327   0.9828  0.8988         230
   DenseNet121 (image-only baseline)    42   Magnetic Resonance Imaging 0.9091   0.9958  0.9799         714
   DenseNet121 (image-only baseline)    42                       Pelvis 0.4203   0.